In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install git+https://github.com/openai/whisper.git
!pip install --upgrade jiwer evaluate

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-obcl_naq
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-obcl_naq
  Resolved https://github.com/openai/whisper.git to commit ba3f3cd54b0e5b8ce1ab3de13e32122d0d5f98ab
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.9 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-

In [ ]:
import sys
sys.path += ['/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/코드/temporal_ensembling']

In [ ]:
# mult값 for문 없는 버전
# medium의 경우에는 38.4gb a100 사용
import train_whisper_ssl
import torch, time, json, random, os
import numpy as np
import pandas as pd

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")

path = '/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/코드/temporal_ensembling/results/'
# 다시 시작할 때 바꾸기
re_start = False # max_val의 0에폭부터 시작할 때는 false
start_epoch = 0 # 끊겼을 때 다시 시작할 에폭
start_max_val = 0.1 # 끊겼을 때 다시 시작할 max_val
# start_mult = 5 # 끊겼을 때 다시 시작할 mult
ver_start = 0 # 끊겼을 때 다시 시작할 버전

meta_data = {
    'model_name' : "openai/whisper-medium",
    'path' : path,
    'base_model_path' : 'whisper_medium_for_temporal_ensembling_200_ver_{}_label_{}.pt',
    'save_model_path' : 'whisper_medium_ssl_200_ver{}_max_val_{}_label_{}.pt',
    're_start': re_start,
    'save_logging_file_name' : 'logging_ssl_medium_200_label_50.json',
    'epochs' : 10,
    'start_epoch': start_epoch,
    'ver_start': ver_start,
    'labeled_data_size' : 50,
    'unlabeled_data_size' : 150,
    'batch_size' : 2,
    'lr' : 2e-5,
    'max_len' : 100,
    'start_max_val' : start_max_val,
    'max_val_lst': [0.1, 0.5, 0.8],
    'alpha' : 0,
    'noise_type' : 'mix5',
    'device' : str(device)
}

def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if re_start == False and ver_start == 0 and start_max_val == 0.1 and start_epoch == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['save_logging_file_name'], 'r') as f:
        all_results = json.load(f)

# for ver in range(5):
#     if ver < ver_start:
#         continue
#     print('====='*4, f'version {ver}', '====='*4)
#     set_seeds(ver)
#     meta_data['seed'] = ver
#     meta_data['ver_start'] = ver
#     ver_start = ver

#     prepro = train_whisper_ssl.CSVPreProcessor(
#         diag_path='/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/senior_train_diag_df.csv',
#         free_path='/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/senior_train_free_df.csv',
#         seed = ver)
#     train_df, valid_df = prepro.collect_data(labeled_df = prepro.free_df, unlabeled_df = prepro.diag_df, label_size = meta_data['labeled_data_size'], unlabel_size = meta_data['unlabeled_data_size'])
#     train_df = prepro.apply_text_preprocessing(train_df)
#     valid_df = prepro.apply_text_preprocessing(valid_df)

def load_prepared_data(ver):
    train_file = f'/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/코드/csv/ssl_train_df_ver_{ver}.csv'
    valid_file = f'/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/코드/csv/ssl_valid_df_ver_{ver}.csv'

    train_df = pd.read_csv(train_file)
    valid_df = pd.read_csv(valid_file)

    return train_df, valid_df

for ver in range(10):
    if ver < ver_start:
        continue
    print('====='*4, f'version {ver}', '====='*4)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    train_df, valid_df = load_prepared_data(ver)

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {'meta_data': meta_data,
                                 'train_data': None,
                                 'train_data_age': None,
                                 'train_data_region': None,
                                 'train_data_gender': None,
                                 'time': None,
                                 'results': {}}
    all_results[str(ver)]['train_data'] = train_df['text'].tolist()
    all_results[str(ver)]['train_data_age'] = train_df['age'].tolist()
    all_results[str(ver)]['train_data_region'] = train_df['region'].tolist()
    all_results[str(ver)]['train_data_gender'] = train_df['gender'].tolist()
    start = time.time()

    for max_val in meta_data['max_val_lst']:
        meta_data['max_val'] = max_val
        if max_val < start_max_val:
            continue
        print('-----'*4, f'max_val {max_val}', '-----'*4)
        if str(max_val) not in all_results[str(ver)]['results'].keys():
            all_results[str(ver)]['results'][str(max_val)] = []
        mult = 1 # 높이면 안 됨 # 1 아래로 줄여보기
        all_results = train_whisper_ssl.train_ssl(ver, max_val, mult, meta_data, train_df, valid_df, all_results)

        start_epoch = 0
        meta_data['start_epoch'] = 0
        re_start = False
        meta_data['re_start'] = False # 새롭게 max_val이 시작될 때는 불러오면 안 되니까

        start_max_val = 0.1
        # meta_data['start_max_val'] = 0.5
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['save_logging_file_name'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

==================== version 0 ====================
-------------------- max_val 0.1 --------------------


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

...

100%|██████████| 100/100 [01:20<00:00,  1.24it/s]


CER: 6.1374, WER: 13.9151, Loss: 0.0251, Supervised Loss: 0.0251, Unsupervised Loss: 0.0000


100%|██████████| 75/75 [00:24<00:00,  3.06it/s]


CER: 30.2068, WER: 65.6667, Loss : 0.3644

Epoch: 10
---------------------


 35%|███▌      | 35/100 [00:29<00:50,  1.28it/s]

In [ ]:
with open('/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/코드/temporal_ensembling/results/logging_ssl_label_800.json', 'r') as f:
    all_results = json.load(f)

In [ ]:
all_results

In [ ]:
all_results[str(0)]['results'].pop('0.8')
all_results

In [ ]:
with open('/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/코드/temporal_ensembling/results/logging_ssl_label_800.json', 'w') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=4)

In [ ]:
# mult값 있는 버전

import train_whisper_ssl
import torch, json, random, os, time
import numpy as np
import whisper

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")

path = '/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/코드/temporal_ensembling/results/'

start_epoch = 0 # 끊겼을 때 다시 시작할 에폭
start_max_val = 0.8 # 끊겼을 때 다시 시작할 max_val
start_mult = 5 # 끊겼을 때 다시 시작할 mult
ver_start = 1 # 끊겼을 때 다시 시작할 버전

meta_data = {
    'model_name' : "openai/whisper-base",
    'path' : path,
    'base_model_path' : 'whisper_base_ssl_baseline.pt',
    'save_model_path' : 'whisper_ssl_ver{}_max_val_{}_mult_{}.pt',
    'baseline_mode': False,
    'save_logging_file_name' : 'logging_ssl.json',
    'epochs' : 3,
    'start_epoch': start_epoch,
    'labeled_data_size' : 5,
    'unlabeled_data_size' : 5,
    'batch_size' : 2,
    'lr' : 2e-5,
    'max_len' : 100,
    'max_val' : start_max_val,
    # 'max_val_lst': [0.3, 0.5, 0.8, 1.0],
    'max_val_lst': [0.8, 1.0, 1.2],
    'mult' : start_mult,
    # 'mult_list' : [1, 2, 3, 4, 5],
    'mult_list' : [4, 5],
    'alpha' : 0.6,
    'noise_type' : 'mix1',
    'device' : str(device)
}

def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if ver_start == 0 and start_epoch == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['save_logging_file_name'], 'r') as f:
        all_results = json.load(f)

for ver in range(5):
    if ver < ver_start:
        continue
    print('====='*4, f'version {ver}', '====='*4)
    set_seeds(ver)
    meta_data['seed'] = ver

    prepro = train_whisper_ssl.CSVPreProcessor(
        diag_path='/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/senior_train_diag_df.csv',
        test_path='/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/senior_test_diag_df.csv',
        free_path='/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/senior_train_free_df.csv',
        seed = ver)
    train_df, valid_df = prepro.collect_data(prepro.diag_df, prepro.free_df, prepro.test_df, meta_data['labeled_data_size'], meta_data['unlabeled_data_size'])
    train_df = prepro.apply_text_preprocessing(train_df)
    valid_df = prepro.apply_text_preprocessing(valid_df)

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {'meta_data': meta_data,
                                 'train_data': None,
                                 'time': None,
                                 'results': {}}

    start = time.time()
    for max_val in meta_data['max_val_lst']:
        if max_val < start_max_val:
            continue
        print('-----'*4, f'max_val {max_val}', '-----'*4)
        if str(max_val) not in all_results[str(ver)]['results']:
            all_results[str(ver)]['results'][str(max_val)] = {}
        meta_data['max_val'] = max_val
        for mult in meta_data['mult_list']:
            if mult < start_mult:
                continue
            print('*****'*4, f'mult {mult}', '*****'*4)
            if str(mult) not in all_results[str(ver)]['results'][str(max_val)]:
                all_results[str(ver)]['results'][str(max_val)][str(mult)] = []
            meta_data['mult'] = mult

            all_results = train_whisper_ssl.train_ssl(ver, max_val, mult, meta_data, train_df, valid_df, all_results)
            meta_data['start_epoch'] = 0 # mult 시작 할 때 설정하면 그 이후로 해당 mult값만 나오는 거 고쳐야 함

    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))
    all_results[str(ver)]['train_data'] = train_df['text'].tolist()

    with open(meta_data['path'] + meta_data['save_logging_file_name'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

==================== version 1 ====================
-------------------- max_val 0.8 --------------------
******************** mult 5 ********************


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(



Epoch: 1
---------------------


  0%|          | 0/5 [00:00<?, ?it/s]/content/drive/MyDrive/연세_수업자료/3학기/기계학습및프로그래밍/코드/temporal_ensembling/train_whisper_ssl.py:156: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_features = torch.tensor(mel, dtype=torch.float32)
100%|██████████| 5/5 [00:02<00:00,  1.78it/s]


Loss: 0.4735, Supervised Loss: 0.3542, Unsupervised Loss: 1.3908


...

CER (best model): 62.8440, Loss : 0.5112

Epoch: 3
---------------------


100%|██████████| 5/5 [00:03<00:00,  1.59it/s]


Loss: 0.1378, Supervised Loss: 0.0943, Unsupervised Loss: 0.1942


100%|██████████| 5/5 [00:01<00:00,  2.64it/s]


CER (best model): 61.4679, Loss : 0.5079


1. time이 0으로 출력
2. epoch 도중 연결 끊겼을 때 다시 불러오기: version1의 경우 0에폭 후 끊고, 1에폭 때 불러왔을 때, 1에폭의 train cer, valid cer, train loss, valid loss가 0에폭의 cer, loss 값보다 모두 큼
3. 2가 안 되는 이유 추론: 모델이 저장 경로의 드라이브에 저장이 안 됨